# 🧠 RAG Pipeline — Configurable Multi-Dataset RAGBench Capstone\n## FinQA • HotpotQA • eManual • traceable experiments\n

---\n# 📌 Section 0 — Multi-Dataset Experiment Configuration\n> Choose one manual profile and optionally override one pipeline parameter at a time. Every run receives a unique artifact fingerprint.\n

In [1]:
# ============================================================
# SECTION 0: MULTI-DATASET CONFIGURATION
# Change only SELECTED_PROFILE and EXPERIMENT_OVERRIDES per run.
# ============================================================
# This is the temporary hard-coded query classifier: select the domain/profile
# yourself.  Later, replace route_query_profile() with an automatic classifier.
SELECTED_PROFILE = "finqa"      # "finqa", "hotpotqa", "emanual", "techqa"
QUERY_ROUTING_MODE = "manual"   # Keep "manual" for this capstone phase
EXPERIMENT_LABEL = "baseline"   # e.g. "bge-base-c320", "no-reranker"
STORAGE_OPTION = 2               # 1 = scoped Google Drive; 2 = manual upload/download
NUM_EVAL_SAMPLES = 50
LLM_MODEL = "llama-3.1-8b-instant"
JUDGE_MODEL = "llama-3.1-8b-instant"

# Starting hypotheses.  Treat these as experiment baselines; tune with evidence.
DATASET_PROFILES = {
    "finqa": {
        "description": "Financial QA with tables and numerical reasoning.",
        "embed_model": "BAAI/bge-small-en-v1.5",
        "chunk_size": 512, "chunk_overlap": 50,
        "top_k_initial": 15, "top_k_final": 5, "use_reranker": True,
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "answer_instructions": "Act as a financial analyst. Preserve figures, units, periods, and calculation steps when the context supports them.",
    },
    "hotpotqa": {
        "description": "Multi-hop open-domain QA; retrieval should cover all supporting facts.",
        "embed_model": "BAAI/bge-base-en-v1.5",
        "chunk_size": 320, "chunk_overlap": 64,
        "top_k_initial": 25, "top_k_final": 6, "use_reranker": True,
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "answer_instructions": "Answer by connecting all necessary supporting facts. Do not infer a bridge fact that is absent from the retrieved context.",
    },
    "emanual": {
        "description": "Technical manuals with long procedural passages.",
        "embed_model": "BAAI/bge-small-en-v1.5",
        "chunk_size": 700, "chunk_overlap": 120,
        "top_k_initial": 20, "top_k_final": 5, "use_reranker": True,
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "answer_instructions": "Answer as a technical-support assistant. Keep ordered procedures, warnings, constraints, and terminology faithful to the retrieved manual text.",
    },
    "techqa": {
        "description": "Technical support QA with product-specific terminology.",
        "embed_model": "BAAI/bge-small-en-v1.5",
        "chunk_size": 640, "chunk_overlap": 100,
        "top_k_initial": 20, "top_k_final": 5, "use_reranker": True,
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "answer_instructions": "Provide precise technical guidance grounded only in the retrieved context; preserve product names, prerequisites, and steps.",
    },
}

# Set any value to None to retain the selected profile's default.
# Example: {"embed_model": "all-MiniLM-L6-v2", "chunk_size": 512, "use_reranker": False}
EXPERIMENT_OVERRIDES = {
    "embed_model": None, "chunk_size": None, "chunk_overlap": None,
    "top_k_initial": None, "top_k_final": None, "use_reranker": None,
    "reranker_model": None,
}

if SELECTED_PROFILE not in DATASET_PROFILES:
    raise ValueError(f"Unknown profile '{SELECTED_PROFILE}'. Choose one of: {list(DATASET_PROFILES)}")
if QUERY_ROUTING_MODE != "manual":
    raise NotImplementedError("Only manual routing is implemented in this notebook version.")

PROFILE = DATASET_PROFILES[SELECTED_PROFILE].copy()
PROFILE.update({k: v for k, v in EXPERIMENT_OVERRIDES.items() if v is not None})
DATASET_NAME = SELECTED_PROFILE
EMBED_MODEL_NAME = PROFILE["embed_model"]
RERANKER_MODEL_NAME = PROFILE["reranker_model"]
USE_RERANKER = PROFILE["use_reranker"]
CHUNK_SIZE = PROFILE["chunk_size"]
CHUNK_OVERLAP = PROFILE["chunk_overlap"]
TOP_K_INITIAL = PROFILE["top_k_initial"]
TOP_K_FINAL = PROFILE["top_k_final"]
ANSWER_INSTRUCTIONS = PROFILE["answer_instructions"]

# One fingerprint per dataset + pipeline setup prevents cache/result collisions.
import hashlib, json, os
PIPELINE_CONFIG = {
    "dataset": DATASET_NAME, "experiment_label": EXPERIMENT_LABEL,
    "embedding_model": EMBED_MODEL_NAME, "reranker_model": RERANKER_MODEL_NAME,
    "use_reranker": USE_RERANKER, "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP, "top_k_initial": TOP_K_INITIAL,
    "top_k_final": TOP_K_FINAL, "llm_model": LLM_MODEL, "judge_model": JUDGE_MODEL,
}
RUN_FINGERPRINT = hashlib.sha1(json.dumps(PIPELINE_CONFIG, sort_keys=True).encode()).hexdigest()[:10]
RUN_ID = f"{DATASET_NAME}__{EXPERIMENT_LABEL}__{RUN_FINGERPRINT}"
EMBED_SAFE = EMBED_MODEL_NAME.replace("/", "_")
LOCAL_CACHE = f"/content/rag_cache/{DATASET_NAME}/{RUN_ID}"
DRIVE_PROJECT_FOLDER = "RAG_Capstone"
DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/{DATASET_NAME}/{RUN_ID}"
ARTIFACT_PREFIX = RUN_ID
CHUNKS_FILE = f"{LOCAL_CACHE}/chunks__{ARTIFACT_PREFIX}.pkl"
FAISS_FILE = f"{LOCAL_CACHE}/faiss__{ARTIFACT_PREFIX}.index"
EMBEDS_FILE = f"{LOCAL_CACHE}/embeddings__{ARTIFACT_PREFIX}.npy"
RESULTS_FILE = f"{LOCAL_CACHE}/results__{ARTIFACT_PREFIX}.csv"
MANIFEST_FILE = f"{LOCAL_CACHE}/manifest__{ARTIFACT_PREFIX}.json"
CHUNKS_FNAME = os.path.basename(CHUNKS_FILE)
FAISS_FNAME = os.path.basename(FAISS_FILE)
EMBEDS_FNAME = os.path.basename(EMBEDS_FILE)
RESULTS_FNAME = os.path.basename(RESULTS_FILE)
os.makedirs(LOCAL_CACHE, exist_ok=True)
with open(MANIFEST_FILE, "w") as f:
    json.dump({"run_id": RUN_ID, "profile": PROFILE, "pipeline_config": PIPELINE_CONFIG}, f, indent=2)

def route_query_profile(query):
    """Manual routing now; future automatic classifier hook."""
    if QUERY_ROUTING_MODE == "manual":
        return DATASET_NAME, "manual selection"
    raise NotImplementedError("Add automatic query classification here after evaluating manual profiles.")

print("=" * 76)
print("MULTI-DATASET RAG CONFIGURATION")
print("=" * 76)
print(f"Manual profile: {DATASET_NAME} | {PROFILE['description']}")
print(f"Experiment run: {RUN_ID}")
print(f"Embedding: {EMBED_MODEL_NAME} | chunks: {CHUNK_SIZE}/{CHUNK_OVERLAP}")
print(f"Retrieval: top-{TOP_K_INITIAL} -> top-{TOP_K_FINAL} | reranker: {USE_RERANKER}")
print(f"Artifacts: {LOCAL_CACHE}")
print("Every evaluation row will contain this run's configuration metadata.")
print("=" * 76)


MULTI-DATASET RAG CONFIGURATION
Manual profile: finqa | Financial QA with tables and numerical reasoning.
Experiment run: finqa__baseline__0b6df1aea7
Embedding: BAAI/bge-small-en-v1.5 | chunks: 512/50
Retrieval: top-15 -> top-5 | reranker: True
Artifacts: /content/rag_cache/finqa/finqa__baseline__0b6df1aea7
Every evaluation row will contain this run's configuration metadata.


---
# 📌 Section 0B — How to run multiple datasets safely

1. In **Section 0**, set `SELECTED_PROFILE` to `finqa`, `hotpotqa`, or `emanual`.
2. Keep the profile defaults for the first run; for a controlled ablation, change only one item in `EXPERIMENT_OVERRIDES` and give it a distinct `EXPERIMENT_LABEL`.
3. Run Sections 1–10. The cache, FAISS index, manifest, and results CSV include a fingerprint of the complete pipeline configuration.
4. Repeat for the next profile. For manual-storage sessions, download every `results__*.csv` file.
5. In Section 11, upload all result CSVs together to compare runs. Do **not** compare configurations with different evaluation sample counts as though they were identical experiments.

**Manual classifier for now:** `SELECTED_PROFILE` is the human-selected query/domain class. The `route_query_profile()` function is the single replacement point for a future automatic classifier.


---
# 📌 Section 1 — Environment Setup
> Install all libraries and imports. Run once per session.


In [2]:
# SECTION 1A: Install Libraries (all in one cell)
!pip install -q datasets sentence-transformers faiss-cpu \
  langchain langchain-community langchain-text-splitters \
  groq gradio scikit-learn huggingface_hub rank_bm25
print("All libraries installed!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All libraries installed!


In [3]:
# SECTION 1B: All Imports
import os, json, time, pickle, re, shutil
import numpy as np
import pandas as pd
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics import mean_squared_error, roc_auc_score
from groq import Groq
from google.colab import userdata, files as colab_files

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    print("GROQ_API_KEY not found - add it via left panel -> Secrets")

groq_client = Groq()
print("All imports ready.")


GROQ_API_KEY loaded from Colab Secrets
All imports ready.


In [4]:
import requests

def get_groq_free_models(api_key: str) -> list[dict]:
    """
    Fetch available models from Groq and show their capacity/status.
    Groq's free tier = all models on the free plan (rate-limited, no billing needed).
    """
    headers = {"Authorization": f"Bearer {api_key}"}
    resp = requests.get("https://api.groq.com/openai/v1/models", headers=headers)
    resp.raise_for_status()

    models = resp.json().get("data", [])

    # Filter only active models (owned by groq = generally available)
    active = [
        {
            "id": m["id"],
            "owned_by": m.get("owned_by", "?"),
            "context_window": m.get("context_window", "?"),
            "active": m.get("active", True),
        }
        for m in models
        if m.get("active", True)
    ]

    return sorted(active, key=lambda x: x["id"])

models = get_groq_free_models(userdata.get("GROQ_API_KEY"))

print(f"✅ {len(models)} models currently available on Groq:\n")
print(f"{'Model ID':<50} {'Owned By':<20} {'Context Window'}")
print("-" * 90)
for m in models:
    print(f"{m['id']:<50} {m['owned_by']:<20} {m['context_window']:,}")

✅ 15 models currently available on Groq:

Model ID                                           Owned By             Context Window
------------------------------------------------------------------------------------------
allam-2-7b                                         SDAIA                4,096
canopylabs/orpheus-arabic-saudi                    Canopy Labs          4,000
canopylabs/orpheus-v1-english                      Canopy Labs          4,000
groq/compound                                      Groq                 131,072
groq/compound-mini                                 Groq                 131,072
llama-3.1-8b-instant                               Meta                 131,072
llama-3.3-70b-versatile                            Meta                 131,072
meta-llama/llama-prompt-guard-2-22m                Meta                 512
meta-llama/llama-prompt-guard-2-86m                Meta                 512
openai/gpt-oss-120b                                OpenAI               13

---
# 📌 Section 2 — Storage Setup
> Runs the storage strategy you chose in Section 0.
> - **Option 1**: Mounts Google Drive, scoped ONLY to `MyDrive/RAG_Capstone/`. All operations stay inside that folder.
> - **Option 2**: No Drive access. Helpers download files to your machine and let you re-upload next session.


In [5]:
# ============================================================
# SECTION 2: Storage Helpers (No Drive)
# ============================================================

def checkpoint_choice(label, local_path):
    """
    Interactive prompt at each checkpoint.
    Returns: 'rebuild', 'cache', or 'upload'
    """
    fname = os.path.basename(local_path)
    has_cache = os.path.exists(local_path)

    print(f"\n{'='*60}")
    print(f"  CHECKPOINT: {label}")
    print(f"{'='*60}")
    if has_cache:
        print(f"  [1] Rebuild from scratch  (deletes cache, reruns everything)")
        print(f"  [2] Load from session cache  <-- '{fname}' found locally")
        print(f"  [3] Upload from your machine  (overwrite with saved file)")
        default = "2"
    else:
        print(f"  [1] Build from scratch  (no cache found)")
        print(f"  [2] (not available - no cache in this session)")
        print(f"  [3] Upload '{fname}' from your machine")
        default = "1"
    print()

    choice = input(f"  Enter 1, 2, or 3  [default={default}]: ").strip()
    if not choice:
        choice = default

    if choice == "1":
        if has_cache:
            os.remove(local_path)
            print(f"  Deleted: {fname}")
        return "rebuild"
    elif choice == "3":
        return "upload"
    elif choice == "2" and has_cache:
        return "cache"
    else:
        print("  Invalid or unavailable. Defaulting to rebuild.")
        return "rebuild"


def upload_file(local_path):
    """Prompt user to upload a file, save to local_path. Returns True on success."""
    fname = os.path.basename(local_path)
    print(f"  Upload '{fname}' (from a previous session):")
    uploaded = colab_files.upload()
    if not uploaded:
        print("  No file uploaded.")
        return False
    for name, data in uploaded.items():
        with open(local_path, "wb") as f:
            f.write(data)
        print(f"  Saved '{name}' as '{fname}'")
        return True
    return False


def download_file(local_path):
    """Download a local file to the user's machine."""
    if os.path.exists(local_path):
        print(f"  Downloading: {os.path.basename(local_path)}")
        colab_files.download(local_path)
    else:
        print(f"  File not found, skipping: {local_path}")


print("Storage helpers ready (no Drive).")
print("Each checkpoint will ask: [1] rebuild / [2] load cache / [3] upload")

Storage helpers ready (no Drive).
Each checkpoint will ask: [1] rebuild / [2] load cache / [3] upload


---
# 📌 Section 3 — Dataset Loading & Exploration
> Fast — no checkpoint needed.


In [6]:
# SECTION 3A: HF Auth + Load Dataset
from huggingface_hub import login
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("HF authenticated via Colab Secrets")
except Exception:
    print("HF_TOKEN not in Secrets - trying interactive...")
    login()

print(f"Loading RAGBench: {DATASET_NAME} (test split)...")
ds = load_dataset("rungalileo/ragbench", DATASET_NAME, split="test")
print(f"Loaded {len(ds)} samples | Columns: {len(ds.column_names)}")


HF authenticated via Colab Secrets
Loading RAGBench: finqa (test split)...


README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

finqa/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 61.1MB            

finqa/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

finqa/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.94MB            

...

finqa/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12502 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1766 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2294 [00:00<?, ? examples/s]

Loaded 2294 samples | Columns: 26


In [7]:
# SECTION 3B: Explore Dataset
s = ds[0]
print("SAMPLE ENTRY")
print(f"  Question:    {s['question'][:100]}...")
print(f"  # Docs:      {len(s['documents'])}")
print(f"  Response:    {s['response'][:100]}...")
print(f"  Adherence:   {s['adherence_score']}")
print(f"  Relevance:   {s['relevance_score']:.4f}")
print(f"  Utilization: {s['utilization_score']:.4f}")
print(f"  Completeness:{s['completeness_score']:.4f}")

all_flat, doc_set = [], set()
for sample in ds:
    for doc in sample["documents"]:
        all_flat.append(doc)
        doc_set.add(doc)
lengths = [len(d) for d in doc_set]
print(f"\nDOCUMENT STATS")
print(f"  Unique docs:  {len(doc_set)}")
print(f"  Dedup ratio:  {len(doc_set)/len(all_flat):.1%}")
print(f"  Avg length:   {np.mean(lengths):.0f} chars")
print(f"  Max length:   {max(lengths)} chars")

print(f"\nGROUND-TRUTH METRIC DISTRIBUTIONS")
for col in ["relevance_score","utilization_score","completeness_score","adherence_score"]:
    vals = [x[col] for x in ds if x[col] is not None]
    if isinstance(vals[0], bool): vals = [1.0 if v else 0.0 for v in vals]
    print(f"  {col:28s}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}")


SAMPLE ENTRY
  Question:    what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?...
  # Docs:      3
  Response:    The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by ...
  Adherence:   True
  Relevance:   0.1111
  Utilization: 0.1111
  Completeness:1.0000

DOCUMENT STATS
  Unique docs:  1097
  Dedup ratio:  16.4%
  Avg length:   1342 chars
  Max length:   6693 chars

GROUND-TRUTH METRIC DISTRIBUTIONS
  relevance_score               mean=0.080  std=0.081
  utilization_score             mean=0.068  std=0.067
  completeness_score            mean=0.914  std=0.217
  adherence_score               mean=0.915  std=0.280


---
# 📌 Section 4 — Document Chunking  ✅ *Checkpointed*
> **Option 1**: Auto-loaded from Drive / saved to Drive.
> **Option 2**: Auto-downloaded after creation; upload prompt at start of new session.


In [8]:
# SECTION 4: Document Chunking (table-aware + interactive checkpoint)

action = checkpoint_choice("Document Chunks", CHUNKS_FILE)

if action == "cache":
    with open(CHUNKS_FILE, "rb") as f:
        chunks, chunk_to_doc, all_documents = pickle.load(f)
    print(f"Loaded from cache: {len(chunks)} chunks from {len(all_documents)} docs")

elif action == "upload":
    if upload_file(CHUNKS_FILE):
        with open(CHUNKS_FILE, "rb") as f:
            chunks, chunk_to_doc, all_documents = pickle.load(f)
        print(f"Loaded from upload: {len(chunks)} chunks")
    else:
        print("Upload failed. Switching to rebuild.")
        action = "rebuild"

if action == "rebuild":
    print(f"Building chunks for '{DATASET_NAME}'...")

    # Collect unique documents
    all_documents, seen = [], set()
    for sample in ds:
        for doc in sample['documents']:
            if doc not in seen:
                seen.add(doc)
                all_documents.append(doc)
    print(f"  Unique documents: {len(all_documents)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
        length_function=len, separators=["\n\n", "\n", ". ", ", ", " ", ""]
    )

    chunks, chunk_to_doc = [], []
    tables_enriched = 0

    for doc_idx, doc in enumerate(all_documents):
        stripped = doc.strip()

        if stripped.startswith('[[') and stripped.endswith(']]'):
            # TABLE: keep intact + prepend natural-language header for better embedding
            try:
                table = ast.literal_eval(stripped)
                col_headers = [h for h in table[0] if h][:6]
                row_labels  = [row[0] for row in table[1:] if row[0]][:6]
                header = (
                    f"Financial data table. "
                    f"Columns: {', '.join(col_headers)}. "
                    f"Rows: {', '.join(row_labels)}. "
                    f"Data: "
                )
                chunks.append(header + stripped)
                tables_enriched += 1
            except Exception:
                chunks.append("Data table: " + stripped)
                tables_enriched += 1
            chunk_to_doc.append(doc_idx)
        else:
            # TEXT: split normally
            for chunk in splitter.split_text(doc):
                chunks.append(chunk)
                chunk_to_doc.append(doc_idx)

    print(f"  Created {len(chunks)} chunks")
    print(f"  Tables enriched with headers: {tables_enriched}")
    print(f"  Avg chunk length: {np.mean([len(c) for c in chunks]):.0f} chars")

    with open(CHUNKS_FILE, "wb") as f:
        pickle.dump((chunks, chunk_to_doc, all_documents), f)
    print("  Saved locally.")
    download_file(CHUNKS_FILE)

print(f"\nTotal chunks: {len(chunks)} | Documents: {len(all_documents)}")


  CHECKPOINT: Document Chunks
  [1] Build from scratch  (no cache found)
  [2] (not available - no cache in this session)
  [3] Upload 'chunks__finqa__baseline__0b6df1aea7.pkl' from your machine

  Enter 1, 2, or 3  [default=1]: 1
Building chunks for 'finqa'...
  Unique documents: 1097
  Created 3866 chunks
  Tables enriched with headers: 379
  Avg chunk length: 383 chars
  Saved locally.
  Downloading: chunks__finqa__baseline__0b6df1aea7.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Total chunks: 3866 | Documents: 1097


---
# 📌 Section 5 — Embedding + FAISS Index  ✅ *Checkpointed*
> Most expensive step (~5 min for finqa). Saved automatically per storage option.


In [9]:
# SECTION 5A: Load Embedding Model
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
embed_dim = embed_model.get_embedding_dimension()
print(f"  Loaded! Dimension: {embed_dim}")


Loading embedding model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Loaded! Dimension: 384


In [10]:
# SECTION 5B: Build or Load FAISS Index

action = checkpoint_choice("FAISS Index", FAISS_FILE)

if action == "cache":
    index = faiss.read_index(FAISS_FILE)
    chunk_embeddings = np.load(EMBEDS_FILE)
    print(f"Loaded from cache: {index.ntotal} vectors x {embed_dim}d")

elif action == "upload":
    print("  Need to upload both files (.index and .npy):")
    ok1 = upload_file(FAISS_FILE)
    ok2 = upload_file(EMBEDS_FILE)
    if ok1 and ok2:
        index = faiss.read_index(FAISS_FILE)
        chunk_embeddings = np.load(EMBEDS_FILE)
        print(f"Loaded from upload: {index.ntotal} vectors")
    else:
        print("Upload incomplete. Switching to rebuild.")
        action = "rebuild"

if action == "rebuild":
    print(f"Embedding {len(chunks)} chunks (3-10 min)...")
    start = time.time()
    chunk_embeddings = embed_model.encode(
        chunks, show_progress_bar=True, batch_size=64, normalize_embeddings=True
    )
    print(f"  Done in {time.time()-start:.1f}s | Shape: {chunk_embeddings.shape}")

    index = faiss.IndexFlatIP(embed_dim)
    index.add(chunk_embeddings.astype("float32"))
    print(f"  {index.ntotal} vectors indexed")

    faiss.write_index(index, FAISS_FILE)
    np.save(EMBEDS_FILE, chunk_embeddings)
    print("  Saved locally.")
    download_file(FAISS_FILE)
    download_file(EMBEDS_FILE)

print(f"FAISS ready: {index.ntotal} vectors x {embed_dim}d")


  CHECKPOINT: FAISS Index
  [1] Build from scratch  (no cache found)
  [2] (not available - no cache in this session)
  [3] Upload 'faiss__finqa__baseline__0b6df1aea7.index' from your machine

  Enter 1, 2, or 3  [default=1]: 1
Embedding 3866 chunks (3-10 min)...


Batches:   0%|          | 0/61 [00:00<?, ?it/s]

  Done in 13.1s | Shape: (3866, 384)
  3866 vectors indexed
  Saved locally.
  Downloading: faiss__finqa__baseline__0b6df1aea7.index


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Downloading: embeddings__finqa__baseline__0b6df1aea7.npy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FAISS ready: 3866 vectors x 384d


---
# 📌 Section 6 — Retrieval Functions


In [11]:
# SECTION 6A: Basic FAISS Retrieval
def retrieve(query, top_k=TOP_K_FINAL):
    """Stage 1: Embed query -> FAISS search -> return top_k chunks."""
    qemb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(qemb, top_k)
    return [chunks[i] for i in idxs[0]], scores[0].tolist(), idxs[0].tolist()

print("retrieve(query, top_k) defined")


retrieve(query, top_k) defined


In [12]:
# SECTION 6B: Two-Stage Retrieval with Reranker

if USE_RERANKER:
    print(f"Loading reranker: {RERANKER_MODEL_NAME}")
    reranker = CrossEncoder(RERANKER_MODEL_NAME)
    print("  Reranker loaded!")
else:
    reranker = None
    print("Reranker disabled (USE_RERANKER = False)")


def retrieve_and_rerank(query, initial_top_k=TOP_K_INITIAL, final_top_k=TOP_K_FINAL):
    """Stage 1: FAISS candidates. Stage 2: CrossEncoder reranks."""
    if reranker is None:
        return retrieve(query, top_k=final_top_k)
    qemb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(qemb, initial_top_k)
    candidates = [(chunks[i], scores[0][r], i) for r, i in enumerate(idxs[0])]
    rscores = reranker.predict([[query, c[0]] for c in candidates])
    reranked = sorted(zip(candidates, rscores), key=lambda x: x[1], reverse=True)
    return ([r[0][0] for r in reranked[:final_top_k]],
            [float(r[1]) for r in reranked[:final_top_k]],
            [r[0][2] for r in reranked[:final_top_k]])

print("retrieve_and_rerank(query) defined")


Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

  Reranker loaded!
retrieve_and_rerank(query) defined


In [13]:
# SECTION 6C: Quick Test
test_q  = ds[0]["question"]
gt_docs = ds[0]["documents"]
print(f"Query: {test_q[:70]}...\n")

r_chunks, r_scores, _ = retrieve(test_q, top_k=5)
print("--- FAISS Only ---")
for rank, (chunk, score) in enumerate(zip(r_chunks, r_scores), 1):
    hit = any(chunk[:80] in gt for gt in gt_docs)
    print(f"  {rank}. {score:.4f} | {"HIT" if hit else "---"} | {chunk[:70]}...")

if USE_RERANKER:
    rr_chunks, rr_scores, _ = retrieve_and_rerank(test_q)
    print("\n--- FAISS + Reranker ---")
    for rank, (chunk, score) in enumerate(zip(rr_chunks, rr_scores), 1):
        hit = any(chunk[:80] in gt for gt in gt_docs)
        print(f"  {rank}. {score:.4f} | {"HIT" if hit else "---"} | {chunk[:70]}...")


Query: what is the rate of return in cadence design systems inc . of an inves...

--- FAISS Only ---
  1. 0.7849 | HIT | . the graph assumes that the value of the investment in our common sto...
  2. 0.7698 | HIT | . , the nasdaq composite index , and s&p 400 information technology ca...
  3. 0.7022 | --- | Data table: [["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29/2012", ...
  4. 0.6686 | --- | . in determining the long-term rate of return for a plan , we consider...
  5. 0.6674 | --- | . as a result , as of january 21 , 2013 , we had repurchased a total o...

--- FAISS + Reranker ---
  1. 5.0747 | HIT | . the graph assumes that the value of the investment in our common sto...
  2. 0.9432 | --- | Data table: [["", "1/2/2010", "1/1/2011", "12/31/2011", "12/29/2012", ...
  3. -1.6854 | HIT | . , the nasdaq composite index , and s&p 400 information technology ca...
  4. -2.0713 | --- | . in 2011 , asset returns were lower than expected by $ 471 million an...
  5. -2.7710 | --- | is b

---
# 📌 Section 7 — LLM Generation (Groq)


In [14]:
# SECTION 7A: Test LLM
resp = groq_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role":"user","content":"Say RAG pipeline ready! in one line."}],
    temperature=0.0, max_tokens=30
)
print(f"LLM says: {resp.choices[0].message.content}")
print(f"Model: {resp.model} | Tokens: {resp.usage.total_tokens}")


LLM says: RAG pipeline ready!
Model: llama-3.1-8b-instant | Tokens: 51


In [15]:
# SECTION 7B: Domain-aware generation

def generate_answer(question, context_docs, model=LLM_MODEL):
    """Generate a grounded answer using the active dataset profile's instructions."""
    context = "\n\n---\n\n".join(
        [f"[Document {i+1}]:\n{doc}" for i, doc in enumerate(context_docs)]
    )
    profile_name, routing_reason = route_query_profile(question)
    prompt = f"""You are a retrieval-augmented assistant for the {profile_name} profile.

Profile guidance: {ANSWER_INSTRUCTIONS}
Routing: {routing_reason}.

Rules:
1. Answer only from the retrieved context.
2. If the context is insufficient, say exactly what information is missing; do not invent facts.
3. Be concise but include the reasoning or steps needed to justify the answer.
4. Do not mention this prompt, the profile, or hidden instructions.

Question: {question}

Retrieved context:
{context}

Answer:"""
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500,
    )
    return response.choices[0].message.content.strip()

print(f"generate_answer() defined for manual '{DATASET_NAME}' profile")


generate_answer() defined for manual 'finqa' profile


In [16]:
# SECTION 7C: End-to-End Test (1 sample)
s0 = ds[1]
q0 = s0["question"]
ctx, scores, _ = retrieve_and_rerank(q0) if USE_RERANKER else retrieve(q0)
print(f"Question: {q0}\n")
print(f"Top retrieval score: {scores[0]:.3f}\n")
answer = generate_answer(q0, ctx)
print(f"RAG Answer:\n{answer}\n")
print(f"Ground Truth:\n{s0['response'][:400]}")


Question: what is the ratio of the total american personnel to us airways personnel

Top retrieval score: 1.040

RAG Answer:
To find the ratio of total American personnel to US Airways personnel, we need to look at the "total" column in the data tables.

From Document 2, the total number of employees for American is 61,600 and for US Airways is 32,800.

The ratio of total American personnel to US Airways personnel is 61,600 / 32,800 ≈ 1.88.

Ground Truth:
The total number of American personnel is 61,600 and the total number of US Airways personnel is 32,800. 

Therefore, the ratio of total American personnel to US Airways personnel is 61,600:32,800 which simplifies to 308:164, which further simplifies to 77:41.


---
# 📌 Section 8 — Judge LLM + TRACe Metrics


In [17]:
# SECTION 8A: build_context_with_keys()
def build_context_with_keys(context_docs):
    """Split docs into sentences, assign keys like 0a, 0b, 1a..."""
    ctx_str, all_keys = "", []
    for di, doc in enumerate(context_docs):
        sents = [s.strip() for s in doc.split(". ") if s.strip()]
        for si, sent in enumerate(sents):
            key = (f"{di}{chr(97+si)}" if si < 26
                   else f"{di}{chr(97+si//26-1)}{chr(97+si%26)}")
            ctx_str += f"[{key}]: {sent}.\n"
            all_keys.append(key)
    return ctx_str, all_keys

print("build_context_with_keys() defined")


build_context_with_keys() defined


In [18]:
# ============================================================
# SECTION 8B: judge_response() — Strict Judge with Retries
# ============================================================

import json
import re
import time

def _safe_judge_output(total, all_keys):
    """Safe fallback when judge fails after all retries."""
    return {
        'all_relevant_sentence_keys': [],
        'all_utilized_sentence_keys': [],
        'sentence_support_information': [],
        'overall_supported': False,
        'total_context_sentences': total,
        'all_context_keys': all_keys,
        'judge_failed': True
    }


def _extract_json(raw_text):
    """Extract valid JSON from LLM output."""
    cleaned = raw_text.strip()
    # Strip Qwen thinking tags (safety net)
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.DOTALL)
    # Strip markdown code fences
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()

    # Try direct parse first
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Find the outermost { ... } block
    brace_start = cleaned.find('{')
    if brace_start == -1:
        raise json.JSONDecodeError("No JSON object found", cleaned, 0)

    depth = 0
    for i in range(brace_start, len(cleaned)):
        if cleaned[i] == '{':
            depth += 1
        elif cleaned[i] == '}':
            depth -= 1
            if depth == 0:
                return json.loads(cleaned[brace_start:i+1])

    return json.loads(cleaned[brace_start:])


def _validate_judge_output(parsed):
    """Validate and normalize the parsed judge JSON."""
    for key in ['all_relevant_sentence_keys', 'all_utilized_sentence_keys']:
        if key not in parsed or not isinstance(parsed[key], list):
            parsed[key] = []
        parsed[key] = [str(k) for k in parsed[key]]

    ssi = parsed.get('sentence_support_information', [])
    if not isinstance(ssi, list):
        ssi = []
    validated_ssi = []
    for item in ssi:
        if isinstance(item, dict):
            validated_ssi.append({
                'response_sentence': str(item.get('response_sentence', '')),
                'supporting_keys': [str(k) for k in item.get('supporting_keys', [])],
                'supported': bool(item.get('supported', False))
            })
    parsed['sentence_support_information'] = validated_ssi
    parsed['overall_supported'] = bool(parsed.get('overall_supported', False))
    return parsed


def judge_response(question, context_docs, response, model=JUDGE_MODEL, max_retries=2):
    """
    Judge LLM evaluates the RAG output.
    Returns TRACe metric inputs: relevant keys, utilized keys,
    sentence support info, overall supported flag.
    """
    ctx_str, all_keys = build_context_with_keys(context_docs)
    total = len(all_keys)

    if total == 0 or not response or not response.strip():
        print("   ⚠️  Empty context or response — skipping judge.")
        return _safe_judge_output(total, all_keys)

    # ── System message (short, cached by Groq) ──────────────
    system_msg = (
        "You are a STRICT evaluation judge for a RAG system. "
        "Return ONLY valid JSON with exactly these keys: "
        "all_relevant_sentence_keys (list of strings), "
        "all_utilized_sentence_keys (list of strings), "
        "sentence_support_information (list of objects with keys: "
        "response_sentence, supporting_keys, supported), "
        "overall_supported (bool). "
        "RELEVANT = sentence directly contains facts needed to answer. "
        "UTILIZED = specific data from sentence appears in the response. "
        "SUPPORTED = every claim in response sentence verified from context. "
        "For finance: calculations are SUPPORTED if source numbers are in context."
    )

    # ── User message (variable content only) ────────────────
    user_msg = (
        f"Context (with sentence keys):\n{ctx_str}\n\n"
        f"Question: {question}\n\n"
        f"Response: {response}\n\n"
        "Return JSON only. Example structure:\n"
        '{"all_relevant_sentence_keys": ["0a", "1b"], '
        '"all_utilized_sentence_keys": ["0a"], '
        '"sentence_support_information": ['
        '{"response_sentence": "The return was 37.9%.", '
        '"supporting_keys": ["0a"], "supported": true}], '
        '"overall_supported": true}'
    )

    # ── Retry loop ───────────────────────────────────────────
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            result = groq_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0.0,
                max_tokens=1024,
                response_format={"type": "json_object"}
            )

            raw = result.choices[0].message.content
            parsed = _extract_json(raw)
            parsed = _validate_judge_output(parsed)

            # Sanity check — retry if all empty (unless it's a refusal)
            is_refusal = ("cannot answer" in response.lower()
                          or "insufficient information" in response.lower())

            if (not is_refusal
                    and len(parsed['all_relevant_sentence_keys']) == 0
                    and len(parsed['sentence_support_information']) == 0
                    and attempt < max_retries):
                print(f"   ⚠️  Judge returned all-empty on attempt {attempt+1} — retrying...")
                time.sleep(2)
                continue

            parsed['total_context_sentences'] = total
            parsed['all_context_keys'] = all_keys
            parsed['judge_failed'] = False
            return parsed

        except json.JSONDecodeError as e:
            last_error = f"JSON parse: {e}"
            if attempt < max_retries:
                print(f"   ⚠️  JSON parse failed (attempt {attempt+1}/{max_retries+1}) — retrying...")
                time.sleep(2 * (attempt + 1))
            continue

        except Exception as e:
            error_str = str(e)
            last_error = error_str

            if "rate_limit" in error_str.lower() or "429" in error_str:
                wait_time = 5 * (attempt + 1)
                print(f"   ⚠️  Rate limited — waiting {wait_time}s (attempt {attempt+1})...")
                time.sleep(wait_time)
                continue

            if attempt < max_retries:
                print(f"   ⚠️  Judge error (attempt {attempt+1}): {error_str[:100]} — retrying...")
                time.sleep(3)
            continue

    print(f"   ❌ Judge failed after {max_retries+1} attempts. Last error: {last_error}")
    return _safe_judge_output(total, all_keys)


print("✅ judge_response() defined")
print(f"   Model:          {JUDGE_MODEL}")
print(f"   Max tokens:     1024  (was 4096)")
print(f"   Response format: json_object (no parse errors)")
print(f"   Prompt split:   system + user (faster)")
print(f"   Retries:        2 with exponential backoff")

✅ judge_response() defined
   Model:          llama-3.1-8b-instant
   Max tokens:     1024  (was 4096)
   Response format: json_object (no parse errors)
   Prompt split:   system + user (faster)
   Retries:        2 with exponential backoff


In [19]:
# SECTION 8C: compute_metrics() - TRACe Formulas
def compute_metrics(j):
    """TRACe formulas (RAGBench paper)."""
    rel   = set(j.get("all_relevant_sentence_keys", []))
    util  = set(j.get("all_utilized_sentence_keys", []))
    total = max(j.get("total_context_sentences", 1), 1)
    ctx_rel  = len(rel) / total
    ctx_util = len(util) / total
    complt   = len(rel & util) / len(rel) if rel else 0.0
    si = j.get("sentence_support_information", [])
    adh = (sum(1 for s in si if s.get("supported", False)) / len(si)
           if si else (1.0 if j.get("overall_supported", False) else 0.0))
    return {"context_relevance": round(ctx_rel,4),
            "context_utilization": round(ctx_util,4),
            "completeness": round(complt,4),
            "adherence": round(adh,4)}

print("compute_metrics(judge_output) defined")
print("  Relevance   = |relevant| / total_context")
print("  Utilization = |utilized| / total_context")
print("  Completeness = (util AND rel) / relevant")
print("  Adherence   = supported_sents / total_response_sents")


compute_metrics(judge_output) defined
  Relevance   = |relevant| / total_context
  Utilization = |utilized| / total_context
  Completeness = (util AND rel) / relevant
  Adherence   = supported_sents / total_response_sents


---
# 📌 Section 9 — Run Full Evaluation  ✅ *Checkpointed*
> **Option 1**: Results CSV auto-saved to Drive and reloaded next session.
> **Option 2**: Results CSV downloaded to your machine; upload prompt next session.


In [20]:
# SECTION 9A: Evaluation Function

def evaluate_pipeline(dataset, num_samples, use_reranker=USE_RERANKER):
    """Retrieve -> Generate -> Judge -> TRACe metrics, retaining full run metadata."""
    results, errors = [], 0
    print(f"Evaluating {num_samples} samples for run: {RUN_ID}")
    print(f"  Profile: {DATASET_NAME} | Reranker: {'ON' if use_reranker else 'OFF'} | LLM: {LLM_MODEL}")
    print(f"  Est. time: {num_samples*5}-{num_samples*12}s\n")

    for i in range(min(num_samples, len(dataset))):
        s = dataset[i]
        q = s["question"]
        print(f"  [{i+1}/{num_samples}] {q[:60]}...")
        try:
            ctx, scores, _ = (retrieve_and_rerank(q) if use_reranker else retrieve(q))
            answer = generate_answer(q, ctx)
            judge = judge_response(q, ctx, answer)
            pred = compute_metrics(judge)
            gt_adh = s.get("adherence_score", 0)
            if isinstance(gt_adh, bool):
                gt_adh = 1.0 if gt_adh else 0.0
            results.append({
                "run_id": RUN_ID, "dataset_name": DATASET_NAME,
                "experiment_label": EXPERIMENT_LABEL, **PIPELINE_CONFIG,
                "index": i, "question": q, "answer": answer,
                "top_retrieval_score": scores[0] if scores else 0,
                "pred_relevance": pred["context_relevance"],
                "pred_utilization": pred["context_utilization"],
                "pred_completeness": pred["completeness"],
                "pred_adherence": pred["adherence"],
                "gt_relevance": s.get("relevance_score", 0),
                "gt_utilization": s.get("utilization_score", 0),
                "gt_completeness": s.get("completeness_score", 0),
                "gt_adherence": gt_adh,
            })
            time.sleep(2)
        except Exception as e:
            print(f"  Sample {i} error: {e}")
            errors += 1
            time.sleep(4)

    print(f"\n{len(results)}/{num_samples} succeeded, {errors} errors.")
    return pd.DataFrame(results)

print("evaluate_pipeline() defined with run metadata")


evaluate_pipeline() defined with run metadata


In [21]:
# SECTION 9B: Run or Load Evaluation Results

action = checkpoint_choice("Evaluation Results", RESULTS_FILE)

if action == "cache":
    results_df = pd.read_csv(RESULTS_FILE)
    print(f"Loaded from cache: {len(results_df)} samples")

elif action == "upload":
    if upload_file(RESULTS_FILE):
        results_df = pd.read_csv(RESULTS_FILE)
        print(f"Loaded from upload: {len(results_df)} samples")
    else:
        print("Upload failed. Switching to rebuild.")
        action = "rebuild"

if action == "rebuild":
    print("Running full evaluation...")
    results_df = evaluate_pipeline(ds, num_samples=NUM_EVAL_SAMPLES)
    results_df.to_csv(RESULTS_FILE, index=False)
    print("Saved locally.")
    download_file(RESULTS_FILE)

cols = ['pred_relevance','pred_utilization','pred_completeness','pred_adherence',
        'gt_relevance','gt_utilization','gt_completeness','gt_adherence']
print("\nRESULTS PREVIEW (first 5)")
print(results_df[cols].head().to_string())


  CHECKPOINT: Evaluation Results
  [1] Build from scratch  (no cache found)
  [2] (not available - no cache in this session)
  [3] Upload 'results__finqa__baseline__0b6df1aea7.csv' from your machine

  Enter 1, 2, or 3  [default=1]: 1
Running full evaluation...
Evaluating 50 samples for run: finqa__baseline__0b6df1aea7
  Profile: finqa | Reranker: ON | LLM: llama-3.1-8b-instant
  Est. time: 250-600s

  [1/50] what is the rate of return in cadence design systems inc . o...
  [2/50] what is the ratio of the total american personnel to us airw...
  [3/50] what portion of total assets acquired of anios are intangibl...
  [4/50] what is the five year total return on the goldman sachs grou...
  [5/50] what is the percentage change in the total carrying amount o...
  [6/50] what is the net change in net revenue during 2015 for enterg...
  [7/50] what was the 2015 total return for the peer group?...
  [8/50] what was the profit margin of printing papers in 2005...
  [9/50] what was the percen

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


RESULTS PREVIEW (first 5)
   pred_relevance  pred_utilization  pred_completeness  pred_adherence  gt_relevance  gt_utilization  gt_completeness  gt_adherence
0          0.2143            0.1429             0.6667          1.0000      0.111111        0.111111              1.0           1.0
1          0.3333            0.3333             1.0000          1.0000      0.040000        0.040000              1.0           1.0
2          0.2143            0.0714             0.3333          1.0000      0.100000        0.050000              0.5           1.0
3          0.2500            0.1250             0.5000          0.3333      0.111111        0.111111              1.0           0.0
4          0.2500            0.1667             0.6667          1.0000      0.050000        0.050000              1.0           1.0


In [22]:
refusals = results_df['answer'].str.lower().str.contains('cannot answer|insufficient').sum()
print(f"Refusal count: {refusals} / {len(results_df)} ({refusals/len(results_df):.0%})")
print(f"Samples with pred_completeness = 0: {(results_df['pred_completeness'] == 0).sum()}")
print(f"Samples with pred_adherence = 0: {(results_df['pred_adherence'] == 0).sum()}")

Refusal count: 0 / 32 (0%)
Samples with pred_completeness = 0: 3
Samples with pred_adherence = 0: 5


---
# 📌 Section 10 — RMSE & AUC-ROC Scores
> Final evaluation numbers for your report.


In [23]:
# SECTION 10: Compute RMSE and AUC-ROC
def compute_evaluation_scores(df, label=""):
    metrics = ["relevance","utilization","completeness","adherence"]
    print("=" * 70)
    print(f"EVALUATION SCORES {label}")
    print("=" * 70)
    print(f"{'Metric':<22} {'RMSE down':>10} {'AUC-ROC up':>12} {'Pred mean':>10} {'GT mean':>8}")
    print("-" * 65)
    rmse_r, auc_r = {}, {}
    for m in metrics:
        pred = df[f"pred_{m}"].values
        gt   = df[f"gt_{m}"].values
        rmse = np.sqrt(mean_squared_error(gt, pred))
        rmse_r[m] = rmse
        gt_bin = (gt >= 0.5).astype(int)
        if len(np.unique(gt_bin)) < 2:
            auc_str, auc_r[m] = "N/A", None
        else:
            try:
                auc = roc_auc_score(gt_bin, pred)
                auc_r[m] = auc
                auc_str = f"{auc:.4f}"
            except Exception:
                auc_str, auc_r[m] = "Error", None
        print(f"{m:<22} {rmse:>10.4f} {auc_str:>12} {np.mean(pred):>10.3f} {np.mean(gt):>8.3f}")
    avg_rmse = np.mean(list(rmse_r.values()))
    valid = [v for v in auc_r.values() if v is not None]
    avg_auc = f"{np.mean(valid):.4f}" if valid else "N/A"
    print("-" * 65)
    print(f"{'AVERAGE':<22} {avg_rmse:>10.4f} {avg_auc:>12}")
    print("=" * 70)
    return rmse_r, auc_r

rmse_results, auc_results = compute_evaluation_scores(
    results_df,
    label=f"[{DATASET_NAME} | {EMBED_MODEL_NAME} | {'reranked' if USE_RERANKER else 'faiss-only'}]"
)


EVALUATION SCORES [finqa | BAAI/bge-small-en-v1.5 | reranked]
Metric                  RMSE down   AUC-ROC up  Pred mean  GT mean
-----------------------------------------------------------------
relevance                  0.3328       0.6129      0.307    0.083
utilization                0.2881          N/A      0.226    0.056
completeness               0.4864       0.7143      0.608    0.844
adherence                  0.5587       0.4955      0.684    0.875
-----------------------------------------------------------------
AVERAGE                    0.4165       0.6076


---
# 📌 Section 11 — Multi-Dataset Comparison
> Run after results exist for at least 2 datasets.
> **Option 1**: Loads CSVs from Drive automatically.
> **Option 2**: Prompts upload for each dataset's results CSV.


In [ ]:
# SECTION 11: Cross-Dataset / Cross-Run Comparison
# Storage Option 1 discovers saved CSVs; Option 2 lets you upload multiple result CSVs.
import glob
from IPython.display import display

if STORAGE_OPTION == 1:
    candidate_files = glob.glob(
        f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/**/results__*.csv", recursive=True
    )
else:
    print("Upload one or more results__*.csv files (including prior FinQA experiments):")
    uploaded = colab_files.upload()
    comparison_dir = "/content/rag_comparison_uploads"
    os.makedirs(comparison_dir, exist_ok=True)
    candidate_files = []
    for name, data in uploaded.items():
        if name.endswith(".csv"):
            path = os.path.join(comparison_dir, name)
            with open(path, "wb") as f:
                f.write(data)
            candidate_files.append(path)

comparison_rows = []
for path in sorted(set(candidate_files)):
    try:
        df = pd.read_csv(path)
        required = {"pred_relevance", "pred_utilization", "pred_completeness", "pred_adherence",
                    "gt_relevance", "gt_utilization", "gt_completeness", "gt_adherence"}
        if df.empty or not required.issubset(df.columns):
            print(f"Skipping {os.path.basename(path)}: not a compatible evaluation CSV.")
            continue
        first = df.iloc[0]
        row = {
            "result_file": os.path.basename(path),
            "run_id": first.get("run_id", "legacy-run"),
            "dataset": first.get("dataset_name", "legacy/unknown"),
            "embedding": first.get("embedding_model", "legacy/unknown"),
            "chunking": f"{first.get('chunk_size', '?')}/{first.get('chunk_overlap', '?')}",
            "retrieval": f"{first.get('top_k_initial', '?')}->{first.get('top_k_final', '?')}",
            "reranker": first.get("use_reranker", "legacy/unknown"),
            "n_samples": len(df),
        }
        for metric in ["relevance", "utilization", "completeness", "adherence"]:
            pred, gt = df[f"pred_{metric}"].values, df[f"gt_{metric}"].values
            row[f"{metric}_rmse"] = round(np.sqrt(mean_squared_error(gt, pred)), 4)
            row[f"{metric}_pred_mean"] = round(float(np.mean(pred)), 3)
            row[f"{metric}_gt_mean"] = round(float(np.mean(gt)), 3)
        comparison_rows.append(row)
    except Exception as e:
        print(f"Skipping {os.path.basename(path)}: {e}")

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows).sort_values(["dataset", "run_id"])
    print("=" * 100)
    print("CROSS-DATASET / CROSS-RUN COMPARISON (lower RMSE = better alignment)")
    print("=" * 100)
    display(comp_df)
    comp_df.to_csv("ragbench_comparison_summary.csv", index=False)
    print("Saved comparison summary: ragbench_comparison_summary.csv")
else:
    print("No compatible result CSVs were provided. Run Section 9 for each profile first.")


---
# 📌 Section 12 — Gradio Demo
> Interactive web app. Run last.


In [24]:
# SECTION 12: Gradio Interactive Demo
import gradio as gr

def rag_ui(question, use_rr, top_k_ui):
    top_k_ui = int(top_k_ui)
    ctx, scores, _ = (retrieve_and_rerank(question, final_top_k=top_k_ui)
                      if use_rr and reranker is not None
                      else retrieve(question, top_k=top_k_ui))
    method = "FAISS + Reranker" if (use_rr and reranker) else "FAISS Only"
    answer = generate_answer(question, ctx)
    ctx_md = ""
    for i, (c, s) in enumerate(zip(ctx, scores), 1):
        ctx_md += f"**Chunk {i}** (score: {s:.4f})\n{c[:400]}{'...' if len(c)>400 else ''}\n\n---\n\n"
    meta = (f"**Dataset:** {DATASET_NAME}  |  **Method:** {method}\n"
            f"**Embed:** {EMBED_MODEL_NAME}  |  **LLM:** {LLM_MODEL}\n"
            f"**Top score:** {scores[0]:.4f}  |  "
            f"**Storage:** Option {STORAGE_OPTION}")
    return answer, ctx_md, meta

with gr.Blocks(title=f"RAG - {DATASET_NAME}") as demo:
    gr.Markdown(
        f"# RAG System - {DATASET_NAME.upper()}\n"
        f"**Dataset:** RAGBench {DATASET_NAME} ({len(ds)} samples) | "
        f"**Chunks:** {len(chunks)} | **Embed:** {EMBED_MODEL_NAME} | **LLM:** {LLM_MODEL}"
    )
    with gr.Row():
        with gr.Column(scale=2):
            q_box = gr.Textbox(label="Question", placeholder="Ask anything...", lines=2)
            with gr.Row():
                rr_cb = gr.Checkbox(label="Use Reranker", value=USE_RERANKER)
                tk_sl = gr.Slider(3, 10, value=TOP_K_FINAL, step=1, label="Chunks")
            btn = gr.Button("Get Answer", variant="primary")
        with gr.Column(scale=1):
            meta_out = gr.Markdown()
    ans_out = gr.Textbox(label="RAG Answer", lines=4, interactive=False)
    with gr.Accordion("Retrieved Context", open=False):
        ctx_out = gr.Markdown()
    gr.Markdown("### Sample Questions")
    gr.Examples([[ds[i]["question"]] for i in range(min(5, len(ds)))], inputs=q_box)
    btn.click(fn=rag_ui, inputs=[q_box, rr_cb, tk_sl], outputs=[ans_out, ctx_out, meta_out])

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d74c57beb8d41f26dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d74c57beb8d41f26dd.gradio.live
